# Combine microns1412 and visp_deltalakes into a single Delta Lake dataset

Auto-discovers all delta tables in both source datasets by scanning for `_delta_log/` directories,
then writes a unified combined dataset to `/scratch/combined_datasets/`.

- Tables that exist at the same relative path in both sources are concatenated.
- Tables unique to one source are copied through as-is.
- Nested sub-tables (e.g. `cellfeatures/csm_cluster_features`) remain independent.

In [ ]:
import os

import pyarrow as pa
import polars as pl
from deltalake import DeltaTable, write_deltalake

In [ ]:
CODEOCEAN_MICRONS = "/data/microns1412"
CODEOCEAN_VISP = "/data/visp_deltalakes"
LOCAL_MICRONS = "../data/microns1412"
LOCAL_VISP = "../data/visp_deltalakes"

MICRONS_ROOT = CODEOCEAN_MICRONS if os.path.exists(CODEOCEAN_MICRONS) else LOCAL_MICRONS
VISP_ROOT = CODEOCEAN_VISP if os.path.exists(CODEOCEAN_VISP) else LOCAL_VISP

OUTPUT_ROOT = "/scratch/combined_datasets"

print(f"Microns root: {MICRONS_ROOT}")
print(f"VISP root:    {VISP_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")

In [ ]:
def discover_delta_tables(root: str) -> dict[str, str]:
    """
    Walk *root* and return {relative_table_path: absolute_table_path}
    for every directory that contains a `_delta_log/` subdirectory.
    """
    root = os.path.abspath(root)
    tables = {}
    for dirpath, dirnames, _ in os.walk(root):
        if "_delta_log" in dirnames:
            rel = os.path.relpath(dirpath, root)
            tables[rel] = dirpath
            dirnames.remove("_delta_log")
    return tables

In [ ]:
microns_tables = discover_delta_tables(MICRONS_ROOT)
visp_tables = discover_delta_tables(VISP_ROOT)

all_rel_paths = sorted(set(microns_tables) | set(visp_tables))

table_sources: dict[str, list[str]] = {}
for rel in all_rel_paths:
    sources = []
    if rel in microns_tables:
        sources.append(microns_tables[rel])
    if rel in visp_tables:
        sources.append(visp_tables[rel])
    table_sources[rel] = sources

print(f"Discovered {len(all_rel_paths)} delta tables:\n")
for rel, srcs in table_sources.items():
    labels = []
    if rel in microns_tables:
        labels.append("microns")
    if rel in visp_tables:
        labels.append("visp")
    print(f"  {rel:50s}  [{', '.join(labels)}]")

In [ ]:
def combine_and_write(rel_path: str, source_paths: list[str], output_root: str) -> int:
    """
    Read delta table(s) from *source_paths*, concatenate if more than one,
    and write a fresh delta table to output_root/rel_path.

    Returns total row count of the combined table.
    """
    tables = []
    partition_cols: list[str] = []

    for src in source_paths:
        dt = DeltaTable(src)
        part_cols = dt.metadata().partition_columns
        if part_cols:
            partition_cols = part_cols
        tables.append(dt.to_pyarrow_table())

    if len(tables) == 1:
        combined = tables[0]
    else:
        combined = pa.concat_tables(tables, promote_options="permissive")

    out_path = os.path.join(output_root, rel_path)
    os.makedirs(out_path, exist_ok=True)

    write_kwargs = dict(mode="overwrite")
    if partition_cols:
        write_kwargs["partition_by"] = partition_cols

    write_deltalake(out_path, combined, **write_kwargs)
    return combined.num_rows

In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for rel_path, source_paths in table_sources.items():
    n_sources = len(source_paths)
    action = "merging" if n_sources > 1 else "copying"
    print(f"{action:8s} {rel_path} ({n_sources} source{'s' if n_sources > 1 else ''}) ... ", end="", flush=True)
    nrows = combine_and_write(rel_path, source_paths, OUTPUT_ROOT)
    print(f"{nrows:,} rows")

print("\nDone.")

## Verification

Read back the combined tables and print shapes to confirm everything was written correctly.

In [ ]:
print(f"{'Table':<50s} {'Rows':>10s} {'Cols':>6s}")
print("-" * 68)

for rel_path in sorted(table_sources):
    out_path = os.path.join(OUTPUT_ROOT, rel_path)
    df = pl.read_delta(out_path)
    print(f"{rel_path:<50s} {df.shape[0]:>10,d} {df.shape[1]:>6d}")

In [ ]:
print("Sample: combined dataset table")
pl.read_delta(os.path.join(OUTPUT_ROOT, "dataset"))

In [ ]:
print("Sample: combined cluster table")
pl.read_delta(os.path.join(OUTPUT_ROOT, "cluster"))

In [ ]:
print("Sample: combined dataitem table")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "dataitem"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
df.head()